# 01 — API sample exploration

Explores dataset `jbjy-vk9h` (SECOP II — Contratos Electrónicos) before designing the extraction and the dimensional model.

Pandas is used here **only for exploration**: this notebook does not transform or load anything. All cleaning happens later in SQL.

**Goals**

1. Column inventory from the API metadata (official types and descriptions).
2. Profile of a sample: nulls, placeholder values, cardinality, examples.
3. Locate the source fields the model needs (entity, supplier, modality, signature date, value, location).
4. Check the formats the SQL cleaning must handle (values, dates, NITs, duplicates).
5. Count the proposed scope (Antioquia, contracts signed 2022–2025) to decide the extraction scope.

**Requirements:** the `secop` Conda environment and a `.env` file in the project root (README, reproduction step 2).

## 0. Setup

In [5]:
import os

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from sodapy import Socrata

# Looks for .env in the current directory and its parents (the notebook runs from notebooks/)
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError('No .env found in this directory or its parents (README, step 2).')
load_dotenv(env_path, override=True)

DOMAIN = os.environ['SOCRATA_DOMAIN']
DATASET = os.environ['SOCRATA_DATASET_ID']
client = Socrata(DOMAIN, os.environ['SOCRATA_APP_TOKEN'], timeout=120)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)


def soql(**params) -> pd.DataFrame:
    """Runs a SoQL query on the server and returns the (small) result as a DataFrame."""
    return pd.DataFrame.from_records(client.get(DATASET, **params))


total = int(soql(select='count(*) AS n')['n'].iloc[0])
print(f'Dataset {DATASET} on {DOMAIN}: {total:,} rows')

Dataset jbjy-vk9h on www.datos.gov.co: 5,687,362 rows


### 0.1 Tests

In [12]:
soql(select='count(*) AS n')            # Total rows

,n
0,5687362


In [13]:
soql(where="fecha_de_firma > '2024-01-01'", limit=100) # Contracts signed since 2024

,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,direcci_n_de_ejecuci_n_del_contrato,criterios_de_sostenibilidad_ambiental,requisitos_ambientales_en_las_especificaciones_t_cnicas,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,uso_de_etiquetado_ecol_gico,obligaciones_posconsumo,fecha_inicio_liquidacion,fecha_fin_liquidacion,ultima_actualizacion
0,CENTRO DE DIAGNOSTICO AUTOMOTOR DE POPAYAN LTDA,800253040,Cauca,Popayán,"Colombia, Cauca , Popayán",Corporación Autónoma,Transporte,Corporación Autónoma,Centralizada,CO1.BDOS.5359886,...,cra 2 No 21 dn 170\nPopayán\nCauca\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN
1,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5356513,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T00:00:00.000,2025-04-30T23:59:00.000,NaN
2,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5352160,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2024-03-01T00:00:00.000,2024-06-29T00:00:00.000,2024-02-29T14:31:25.745
3,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5354459,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-04-30T11:59:00.000,NaN
4,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5349839,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-04-30T11:59:00.000,NaN
5,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5350223,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T00:00:00.000,2025-04-30T00:00:00.000,2026-07-15T18:01:16.350
6,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5354586,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-03-20T11:59:00.000,2025-10-30T14:55:21.816
7,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5354194,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-02-07T11:59:00.000,2025-10-30T14:34:58.301
8,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5350207,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T00:00:00.000,2025-04-30T00:00:00.000,2024-12-06T15:58:44.702
9,EMPRESA SOCIAL DEL ESTADO E.S.E HOSPITAL LA MILAGROSA,890702476,Tolima,Villarrica,"Colombia, Tolima , Villarrica",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5360601,...,Cra 4 Calle 1 #

## 1. Column inventory (API metadata)

The API returns every value as text in JSON. The metadata gives the official type of each column, which tells us what conversion each one will need in SQL.

In [3]:
meta = client.get_metadata(DATASET)
columns = pd.DataFrame(
    [
        {
            'field': c.get('fieldName'),
            'name': c.get('name'),
            'api_type': c.get('dataTypeName'),
            'description': (c.get('description') or '').strip()[:120],
        }
        for c in meta['columns']
        if not c.get('fieldName', '').startswith(':')  # skip system columns
    ]
)
types = dict(zip(columns['field'], columns['api_type']))

updated = pd.to_datetime(meta.get('rowsUpdatedAt'), unit='s')
print(f'{len(columns)} columns; data last updated: {updated}')
columns['api_type'].value_counts()

95 columns; data last updated: 2026-11-06 13:27:53


api_type
text             64
number           19
calendar_date    11
url               1
Name: count, dtype: int64

In [19]:
columns

,field,name,api_type,description
0,nombre_entidad,Nombre Entidad,text,Nombre de la entidad del estado que publica el contrato
1,nit_entidad,Nit Entidad,number,NIT de la entidad del estado que publica el contrato
2,departamento,Departamento,text,Departamento en el cual se registró la entidad del estado que publica el con...
3,ciudad,Ciudad,text,Ciudad en el cual se registró la entidad del estado que publica el contrato
4,localizaci_n,Localización,text,Ubicación completa de la entidad del estado que publica el contrato
5,orden,Orden,text,Orden entidad del estado que publica el contrato
6,sector,Sector,text,Sector entidad del estado que publica el contrato
7,rama,Rama,text,Rama del estado de la entidad que publica el contrato
8,entidad_centralizada,Entidad Centralizada,text,Define si la entidad es descentralizada o centralizada
9,proceso_de_compra,Proceso de Compra,text,Identificador del proceso de compra publicado


### 1.2 Column inventory (SoQL query):

In [20]:
df = soql(limit=100)
schema = (
    pd.DataFrame({
        "columna": df.columns,
        "tipo":   [str(t) for t in df.dtypes],
    })
    .sort_values("columna")
    .reset_index(drop=True)
)
schema

,columna,tipo
0,ciudad,str
1,codigo_de_categoria_principal,str
2,codigo_entidad,str
3,codigo_proveedor,str
4,condiciones_de_entrega,str
5,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,str
6,criterios_de_sostenibilidad_ambiental,str
7,departamento,str
8,descripcion_del_proceso,str
9,descripcion_documentos_tipo,str


## 2. Sample

5,000 rows ordered by the internal `:id`. The order is stable, so the sample is reproducible, but it is **not random**: treat proportions from it as indicative only. The full-dataset counts in section 6 come from the server.

The API omits fields that are null in a row, so the sample is reindexed to the full column list.

In [25]:
SAMPLE_SIZE = 5_000
sample = soql(limit=SAMPLE_SIZE, order=':id').reindex(columns=columns['field'])
print(sample.shape)
sample.head(4)

(5000, 95)


field,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,criterios_de_sostenibilidad_ambiental,requisitos_ambientales_en_las_especificaciones_t_cnicas,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,uso_de_etiquetado_ecol_gico,obligaciones_posconsumo,fecha_inicio_obligaciones_posconsumo,fecha_fin_obligaciones_posconsumo,fecha_inicio_reversi_n,fecha_fin_reversi_n
0,DANE - DIRECCION TERRITORIAL NORTE,899999027,Atlántico,No Definido,"Colombia, Atlántico , No Definido",Nacional,Información Estadística,Ejecutivo,Centralizada,CO1.BDOS.6658824,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN
1,GOBERNACION DEL QUINDIO ARMENIA,890001639,Quindío,Armenia,"Colombia, Quindío, Armenia",Territorial,No aplica/No pertenece,Ejecutivo,Centralizada,CO1.BDOS.7744023,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN
2,IGAC Sede Central *,899999004,Distrito Capital de Bogotá,No Definido,"Colombia, Bogotá, No Definido",Nacional,Información Estadística,Ejecutivo,Centralizada,CO1.BDOS.9606774,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN
3,CORPORACION AUTONOMA REGIONAL DE LA GUAJIRA,892115314,La Guajira,Riohacha,"Colombia, La Guajira , Riohacha",Corporación Autónoma,Ambiente y Desarrollo Sostenible,Corporación Autónoma,Centralizada,CO1.BDOS.7704855,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN


## 3. Column profile

Open-data sources often fill missing fields with text such as `No Definido` or `No Aplica`. These are nulls in disguise, so the profile counts them separately (`placeholder_pct`). Extend `PLACEHOLDERS` if the examples show others.

In [29]:
PLACEHOLDERS = {'', '-', 'n/a', 'na', 'null', 'none', 'no definido', 'no aplica',
                'sin descripcion', 'sin descripción'}


def profile(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    rows = []
    for col in df.columns:
        s = df[col]
        text = s.dropna().astype(str).str.strip()
        is_placeholder = text.str.lower().isin(PLACEHOLDERS)
        real = text[~is_placeholder]
        rows.append({
            'field': col,
            'api_type': types.get(col),
            'null_pct': round(100 * s.isna().mean(), 1),
            'placeholder_pct': round(100 * is_placeholder.sum() / n, 1),
            'distinct': real.nunique(),
            'examples': ' | '.join(real.drop_duplicates().head(3).str[:40]),
        })
    return pd.DataFrame(rows).set_index('field')


prof = profile(sample)
prof

,api_type,null_pct,placeholder_pct,distinct,examples
field,,,,,
nombre_entidad,text,0.0,0.0,1262,DANE - DIRECCION TERRITORIAL NORTE | GOBERNACION DEL QUINDIO ARMENIA | IGAC ...
nit_entidad,number,0.0,0.0,1035,899999027 | 890001639 | 899999004
departamento,text,0.0,2.0,32,Atlántico | Quindío | Distrito Capital de Bogotá
ciudad,text,0.0,21.8,354,Armenia | Riohacha | Duitama
localizaci_n,text,0.0,0.0,426,"Colombia, Atlántico , No Definido | Colombia, Quindío, Armenia | Colombia, ..."
orden,text,0.0,0.0,3,Nacional | Territorial | Corporación Autónoma
sector,text,0.0,0.0,24,Información Estadística | No aplica/No pertenece | Ambiente y Desarrollo Sos...
rama,text,0.0,0.0,4,Ejecutivo | Corporación Autónoma | Legislativo
entidad_centralizada,text,0.0,0.0,2,Centralizada | Descentralizada


In [30]:
# Columns that are almost always empty in the sample: candidates to leave out of the model
prof[prof['null_pct'] + prof['placeholder_pct'] >= 90]

,api_type,null_pct,placeholder_pct,distinct,examples
field,,,,,
puntos_del_acuerdo,text,0.0,99.5,3,ReformaRuralIntegral | FinConflicto | VictimasConflicto
pilares_del_acuerdo,text,0.0,99.5,4,OSPRUDS | RFARCVC | JusticiaYVerdad
fecha_de_notificaci_n_de_prorrogaci_n,calendar_date,90.3,0.0,406,2023-12-16T04:59:00.000 | 2024-02-01T04:00:00.000 | 2026-12-01T04:59:00.000
descripcion_documentos_tipo,text,0.0,99.6,6,Catastro multipropósito - Contratos inte | Infraestructura de transporte | C...
criterios_de_sostenibilidad_ambiental,text,0.0,98.6,2,No | Sí
requisitos_ambientales_en_las_especificaciones_t_cnicas,text,0.0,99.9,2,Sí | No
criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,text,0.0,99.9,2,Sí | No
obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,text,0.0,99.9,2,Sí | No
uso_de_etiquetado_ecol_gico,text,0.0,99.9,1,No


## 4. Source fields for the model

`CANDIDATES` maps each role in the dimensional model to the source field expected to fill it. The table checks whether each one exists and shows its profile. If a field is missing or looks wrong, use `find()` to search the inventory and **correct the mapping here**. The confirmed names are what the staging table and the model will use.

In [ ]:
CANDIDATES = {
    'contract id': 'id_contrato',
    'entity NIT': 'nit_entidad',
    'entity name': 'nombre_entidad',
    'entity level': 'orden',
    'entity sector': 'sector',
    'supplier document type': 'tipodocproveedor',
    'supplier document': 'documento_proveedor',
    'supplier name': 'proveedor_adjudicado',
    'selection modality': 'modalidad_de_contratacion',
    'contract type': 'tipo_de_contrato',
    'contract status': 'estado_contrato',
    'signature date': 'fecha_de_firma',
    'start date': 'fecha_de_inicio_del_contrato',
    'end date': 'fecha_de_fin_del_contrato',
    'contract value': 'valor_del_contrato',
    'department': 'departamento',
    'city': 'ciudad',
    'last update': 'ultima_actualizacion',
}

mapping = pd.DataFrame(
    [{'role': role, 'field': f, 'exists': f in types} for role, f in CANDIDATES.items()]
).set_index('role')
display(mapping.join(prof, on='field'))

missing = mapping.loc[~mapping['exists'], 'field'].tolist()
print('All candidate fields exist.' if not missing else f'Not found, search with find(): {missing}')

In [ ]:
def find(*words: str) -> pd.DataFrame:
    """Columns whose field name or display name contains any of the words."""
    pattern = '|'.join(words)
    hit = (columns['field'].str.contains(pattern, case=False)
           | columns['name'].str.contains(pattern, case=False))
    return columns.loc[hit, ['field', 'name', 'api_type']].join(
        prof[['null_pct', 'examples']], on='field')


for words in [('depart', 'munic', 'ciudad', 'dane', 'localiz'),
              ('valor',),
              ('fecha',),
              ('proveedor', 'documento'),
              ('adicion', 'dias', 'duraci')]:
    print('Search:', ', '.join(words))
    display(find(*words))

## 5. Formats the SQL cleaning must handle

Conversions here use `errors='coerce'` only to **measure** how many values would fail; the real conversion is done later in SQL.

In [ ]:
# Monetary value
VALUE = CANDIDATES['contract value']
v = sample[VALUE].dropna().astype(str).str.strip()
num = pd.to_numeric(v, errors='coerce')
print(f'{VALUE}: {len(v):,} non-null | {num.isna().sum():,} not numeric | '
      f'{(num == 0).sum():,} zero | {(num < 0).sum():,} negative')
print('Non-numeric examples:', v[num.isna()].unique()[:5])
num.describe(percentiles=[0.5, 0.9, 0.99]).map(lambda x: f'{x:,.0f}')

In [ ]:
# Dates
for role in ['signature date', 'start date', 'end date', 'last update']:
    col = CANDIDATES[role]
    s = sample[col].dropna().astype(str).str.strip()
    if s.empty:
        print(f'{col}: empty in the sample')
        continue
    d = pd.to_datetime(s, errors='coerce', format='ISO8601')
    print(f'{col}: {len(s):,} non-null | {d.isna().sum():,} unparseable | '
          f'range {d.min()} -> {d.max()} | raw example {s.iloc[0]!r}')

In [ ]:
# Document (NIT) formats: digits become 9 and letters become A, so each distinct shape is one format
def shape(s: pd.Series) -> pd.Series:
    return (s.str.replace(r'\d', '9', regex=True)
             .str.replace(r'[^\W\d_]', 'A', regex=True))


display(sample[CANDIDATES['supplier document type']].value_counts(dropna=False).head(10))
for role in ['supplier document', 'entity NIT']:
    col = CANDIDATES[role]
    s = sample[col].dropna().astype(str).str.strip()
    print(f'{col}: most common formats')
    display(shape(s).value_counts().head(10))

In [ ]:
# Duplicates of the contract key (versions of the same contract)
ID = CANDIDATES['contract id']
dup = sample[ID].duplicated(keep=False)
print(f'{ID}: {sample[ID].nunique():,} distinct in {len(sample):,} rows | {dup.sum():,} rows share an id')
show = [ID, CANDIDATES['contract status'], CANDIDATES['contract value'], CANDIDATES['last update']]
sample.loc[dup, show].sort_values(ID).head(10)

In [ ]:
# Categorical fields: values and frequencies in the sample
for role in ['selection modality', 'contract type', 'contract status', 'entity level']:
    col = CANDIDATES[role]
    print(col)
    display(sample[col].value_counts(dropna=False).head(20))

## 6. Scope count (server-side SoQL)

These queries aggregate on the server over the whole dataset; only the small results are downloaded. `date_extract_y` requires the signature date to be a `calendar_date` column (check `api_type` in section 1).

In [ ]:
DEPT = CANDIDATES['department']
DATE = CANDIDATES['signature date']

by_dept = soql(select=f'{DEPT}, count(*) AS n', group=DEPT, order='count(*) DESC', limit=200)
by_dept['n'] = by_dept['n'].astype(int)
by_dept['pct'] = (100 * by_dept['n'] / total).round(2)
by_dept

In [ ]:
# The exact spelling of the department as stored in the source (it may differ in case or accents)
matches = by_dept[by_dept[DEPT].str.contains('antioquia', case=False, na=False)]
display(matches)
SCOPE_DEPT = matches[DEPT].iloc[0]
print(f'Department value used for the scope: {SCOPE_DEPT!r}')

In [ ]:
# Contracts per signature year in the department (a null year = no signature date)
year = f'date_extract_y({DATE})'
by_year = soql(select=f'{year} AS anio, count(*) AS n', where=f"{DEPT} = '{SCOPE_DEPT}'",
               group=year, order=year, limit=200)
by_year['n'] = by_year['n'].astype(int)
by_year

In [ ]:
START, END = '2022-01-01T00:00:00', '2026-01-01T00:00:00'  # END is exclusive
scope_where = f"{DEPT} = '{SCOPE_DEPT}' AND {DATE} >= '{START}' AND {DATE} < '{END}'"
scope_n = int(soql(select='count(*) AS n', where=scope_where)['n'].iloc[0])
no_date = int(soql(select='count(*) AS n', where=f"{DEPT} = '{SCOPE_DEPT}' AND {DATE} IS NULL")['n'].iloc[0])

PAGE = 50_000
bytes_per_row = sample.memory_usage(deep=True).sum() / len(sample)
print(f'Proposed scope ({SCOPE_DEPT}, signed 2022-2025): {scope_n:,} rows '
      f'({100 * scope_n / total:.1f}% of the dataset)')
print(f'Rows in {SCOPE_DEPT} without signature date (outside any date range): {no_date:,}')
print(f'Extraction: about {-(-scope_n // PAGE)} pages of {PAGE:,} rows; '
      f'roughly {scope_n * bytes_per_row / 1e9:.1f} GB as an in-memory DataFrame (Parquet on disk is much smaller)')

In [ ]:
client.close()

## 7. Findings

*Pending — fill in after running the notebook.*

- **Confirmed source fields:** *pending* (final `CANDIDATES` mapping).
- **Formats to clean in SQL:** *pending* (values, dates, document formats, placeholders).
- **Duplicates:** *pending* (does `id_contrato` repeat, and why).
- **Scope decision:** *pending* (row count for Antioquia 2022–2025 and whether to keep it or narrow it).